In [ ]:
import time
import pandas as pd
import os
import joblib

In [ ]:
# feature columns
# feature columns
X_cols = [
    "hour","dayofweek","is_weekend","month",
    # "lag_24","rolling_24",
    "airTemperature", "dewTemperature", "windSpeed",  # weather
    # "temp_lag_1h","dewTemperature_lag_1h", "windSpeed_lag_1h",
    # "sqft", 
    "sqm", 
    "primaryspaceusage", "site_id", "building_id",
    # 'chilled_delta', 'hot_delta',
    "Chilledwater", "Hotwater",
    'chilled_per_sqm', 'hot_per_sqm',
    'chilled_ratio_e', 'hot_ratio_e',
    # 'chilled_ratio_e_lag1', 'hot_ratio_e_lag1',
    'thermal_balance',
    'thermal_load', 'thermal_load_per_sqm'
]

In [ ]:
model_dir = "models_1578_csv"
os.makedirs(model_dir, exist_ok=True)

In [ ]:
def load_data(path):
    data_df = pd.read_csv(path)
    return data_df

# TRAINING


In [ ]:

data_encoded = "data_1578_csv/train_encode.csv"
data_frame = load_data(data_encoded)

In [ ]:
# data_frame[["primaryspaceusage", "site_id", "buiding_id"]].head()
data_frame

In [ ]:
# import time
# import xgboost as xgb
# import lightgbm as lgb

# forecast_horizon = 24

# for h in range(forecast_horizon):
#     times = time.time()
#     model = lgb.LGBMRegressor(
#         device="gpu",
#         n_estimators=100
#     )# defaut 100
#     # model.fit(data_frame[X_cols], data_frame[f"target_t+{h+1}"])

#     joblib.dump(model, f"{model_dir}/model_hour_{h+1}.pkl")
#     print(f"Training time model {h} {time.time() - times}:", )

In [ ]:
import time
import xgboost as xgb

forecast_horizon = 24
times = time.time()
for h in range(forecast_horizon):
    model = xgb.XGBRegressor(
        tree_method="hist",   # ⚠️ BẮT BUỘC
        device="cuda",        # ✅ bật GPU

        n_estimators=1000,
        max_depth=6,
        learning_rate=0.01,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="reg:squarederror",
        random_state=42
    )
    
    target_col = f"target_t+{h+1}"
    df_train = data_frame[X_cols + [target_col]].dropna()
    print(f"Training model for horizon {h+1}, training samples: {len(df_train)}")
    X_train = df_train[X_cols]
    y_train = df_train[target_col]
    
    model.fit(X_train, y_train)
    joblib.dump(model, f"{model_dir}/model_hour_{h+1}.pkl")
print("Training time:", time.time() - times)